In [ ]:
# Step 1: Install all necessary libraries
!pip install -U timm ultralytics scikit-image torch torchvision pandas tqdm scikit-learn


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.1/63.1 kB 3.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 26.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 22.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 899.7/899.7 MB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 594.3/594.3 MB 865.0 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 80.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 MB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 954.8/954.8 kB 52.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.1/193.1 MB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 49.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.6/63.6 MB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 267.5/2

In [ ]:

import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.models.detection as detection
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, Dataset
from ultralytics import YOLO
from skimage.feature import local_binary_pattern
import cv2
import timm

from PIL import Image, UnidentifiedImageError
import numpy as np
import pandas as pd
import os
import shutil
from tqdm import tqdm

# Import sklearn tools for PCA and Scaling
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

from google.colab import drive

# --- Configuration ---
DRIVE_BASE_PATH = '/content/drive/MyDrive/Lung Cancer'
LOCAL_DATA_PATH = '/content/Lung Cancer local' # Local path for stability
OUTPUT_PATH = os.path.join(DRIVE_BASE_PATH, 'output_standardized_pca2048') # Save to a new folder

# This is the TARGET number of features.
# Models with MORE features will be reduced to this.
# Models with FEWER features will be kept as-is.
TARGET_FEATURES = 2048
RANDOM_STATE = 42

CLASS_NAMES = ['Bengin', 'Normal', 'Malignant']
EMOTION_LABELS = {name: i for i, name in enumerate(CLASS_NAMES)}

BATCH_SIZE = 32
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
VALID_EXTENSIONS = ('.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff')

# --- Custom Dataset Class (Handles both DL and LBP) ---
class FERDataset(Dataset):
    """Custom Dataset for loading FER-2013 images."""
    def __init__(self, root_dir, transform=None, is_lbp=False):
        self.root_dir = root_dir
        self.transform = transform
        self.is_lbp = is_lbp
        self.image_paths = []
        self.labels = []

        if not os.path.isdir(root_dir):
             print(f"ERROR: Data directory not found at {root_dir}")
             raise FileNotFoundError(f"Data directory not found: {root_dir}")

        for folder, label in EMOTION_LABELS.items():
            folder_path = os.path.join(root_dir, folder)
            if os.path.isdir(folder_path):
                for img_name in os.listdir(folder_path):
                    if img_name.lower().endswith(VALID_EXTENSIONS):
                        self.image_paths.append(os.path.join(folder_path, img_name))
                        self.labels.append(label)
            else:
                print(f"  Warning: Folder not found for class '{folder}' at {folder_path}")

        if not self.image_paths:
             print(f"ERROR: No valid images found. Check folder names in {root_dir}")
        else:
            print(f"Found {len(self.image_paths)} valid image files in {root_dir}.")

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        label = self.labels[idx]
        try:
            if self.is_lbp:
                image = cv2.imread(img_path)
                if image is None: raise IOError(f"OpenCV could not read image: {img_path}")
                return image, label, img_path
            else:
                image = Image.open(img_path).convert('RGB')
                if self.transform:
                    image = self.transform(image)
                return image, label, img_path
        except (UnidentifiedImageError, IOError) as e:
            # print(f"\nWarning: Skipping corrupt/unreadable file: {img_path}. Error: {e}")
            return None, -1, "invalid_path"
        except Exception as e:
            # print(f"\nWarning: Skipping file: {img_path}. Error: {e}")
            return None, -1, "invalid_path"

def collate_fn(batch):
    """Custom collate function to filter out None items from dataset errors."""
    batch = list(filter(lambda x: x[0] is not None, batch))
    if not batch: return None
    return torch.utils.data.dataloader.default_collate(batch)

# --- Model Creation Functions ---
def create_resnet50_extractor():
    print("Loading ResNet-50...")
    model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
    model = nn.Sequential(*list(model.children())[:-1]) # 2048 features
    return model, "resnet50_features", None

def create_retinanet_extractor():
    print("Loading RetinaNet (ResNet-50 FPN) backbone...")
    model = detection.retinanet_resnet50_fpn(weights=detection.RetinaNet_ResNet50_FPN_Weights.COCO_V1)
    return model.backbone, "retinanet_features", None

def create_vgg16_extractor():
    print("Loading VGG16 (4096 features)...")
    model = models.vgg16(weights=models.VGG16_Weights.IMAGENET1K_V1)
    model.classifier = nn.Sequential(*list(model.classifier.children())[:-1])
    return model, "vgg16_features", None

def create_mobilenetv3_extractor():
    print("Loading MobileNetV3-Large...")
    model = models.mobilenet_v3_large(weights=models.MobileNet_V3_Large_Weights.IMAGENET1K_V1)
    model.classifier[-1] = nn.Identity() # 960 features
    return model, "mobilenetv3_features", None

def create_convnext_extractor():
    print("Loading ConvNeXt-Base...")
    model = timm.create_model('convnext_base.fb_in22k_ft_in1k', pretrained=True)
    model.head.fc = nn.Identity() # 1024 features
    data_config = timm.data.resolve_data_config({}, model=model)
    transform = timm.data.create_transform(**data_config, is_training=False)
    return model, "convnext_features", transform

def create_efficientnetv2_m_extractor():
    print("Loading EfficientNetV2-M...")
    model = timm.create_model('tf_efficientnetv2_m.in21k_ft_in1k', pretrained=True)
    model.classifier = nn.Identity() # 1280 features
    data_config = timm.data.resolve_data_config({}, model=model)
    transform = timm.data.create_transform(**data_config, is_training=False)
    return model, "efficientnet_v2_m_features", transform

# --- Feature Extraction Functions ---
def extract_dl_features(data_loader, model, model_name):
    """Extracts features for standard PyTorch models (ResNet, VGG, MobileNet, EffNet, ConvNeXt)."""
    model.eval()
    features_list, labels_list, paths_list = [], [], []
    with torch.no_grad():
        for batch in tqdm(data_loader, desc=f"Extracting {model_name} Features"):
            if batch is None: continue
            inputs, labels, paths = batch
            valid_idx = labels != -1
            if not torch.any(valid_idx): continue

            inputs = inputs[valid_idx].to(DEVICE)
            labels = labels[valid_idx]
            paths = [paths[i] for i, v in enumerate(valid_idx) if v]

            outputs = model(inputs)
            features_list.append(outputs.cpu().numpy().reshape(outputs.shape[0], -1))
            labels_list.extend(labels.cpu().numpy())
            paths_list.extend(paths)
    if not features_list: return None, None, None
    return np.vstack(features_list), np.array(labels_list), paths_list

def extract_retinanet_features(data_loader, model):
    """Extracts features specifically for the RetinaNet backbone."""
    model.eval()
    pool = nn.AdaptiveAvgPool2d(1)
    features_list, labels_list, paths_list = [], [], []
    with torch.no_grad():
        for batch in tqdm(data_loader, desc="Extracting RetinaNet Features"):
            if batch is None: continue
            inputs, labels, paths = batch
            valid_idx = labels != -1
            if not torch.any(valid_idx): continue

            inputs = inputs[valid_idx].to(DEVICE)
            labels = labels[valid_idx]
            paths = [paths[i] for i, v in enumerate(valid_idx) if v]

            feature_maps = model(inputs)
            outputs = feature_maps.get('pool', feature_maps[list(feature_maps.keys())[-1]])
            pooled = pool(outputs)
            features_list.append(pooled.cpu().numpy().reshape(pooled.shape[0], -1))
            labels_list.extend(labels.cpu().numpy())
            paths_list.extend(paths)
    if not features_list: return None, None, None
    return np.vstack(features_list), np.array(labels_list), paths_list

def extract_yolo_features(dataset):
    """Extracts features using YOLOv8's embed function (processes paths)."""
    model = YOLO('yolov8n-cls.pt')
    features_list, labels_list, paths_list = [], [], []
    for img_path, label in tqdm(zip(dataset.image_paths, dataset.labels), total=len(dataset), desc="Extracting YOLOv8 Features"):
        try:
            results = model.embed(img_path, verbose=False)
            features_list.append(results[0].cpu().numpy())
            labels_list.append(label)
            paths_list.append(img_path)
        except Exception as e:
            print(f"\nWarning: YOLO could not process {img_path}. Error: {e}")
    if not features_list: return None, None, None
    return np.vstack(features_list), np.array(labels_list), paths_list

def extract_lbp_features(dataset):
    """Extracts LBP features using OpenCV (CPU based)."""
    features_list, labels_list, paths_list = [], [], []
    P, R = 8, 1
    for item in tqdm(dataset, desc="Extracting LBP Features"):
        image, label, path = item
        if image is None: continue
        gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
        resized = cv2.resize(gray, (224, 224))
        lbp = local_binary_pattern(resized, P, R, method="uniform")
        (hist, _) = np.histogram(lbp.ravel(), bins=np.arange(0, P + 3), range=(0, P + 2))
        hist = hist.astype("float")
        hist /= (hist.sum() + 1e-6)
        features_list.append(hist)
        labels_list.append(label)
        paths_list.append(path)
    if not features_list: return None, None, None
    return np.array(features_list), np.array(labels_list), paths_list

def save_csv(features, labels, paths, file_prefix, dataset_type, n_components):
    """Saves the final features to a CSV file."""
    if features is None:
        print(f"No valid features were extracted for {file_prefix} ({dataset_type}). Skipping save.")
        return

    print(f"Final features shape: {features.shape}")

    # Create the correct filename
    if n_components == features.shape[1]:
        # This means PCA was skipped
        output_filename = os.path.join(OUTPUT_PATH, f'{file_prefix}_{dataset_type}.csv')
        feature_cols = [f'feature_{i}' for i in range(features.shape[1])]
    else:
        # This means PCA was applied
        output_filename = os.path.join(OUTPUT_PATH, f'{file_prefix}_pca{n_components}_{dataset_type}.csv')
        feature_cols = [f'pca_feature_{i}' for i in range(features.shape[1])]

    df = pd.DataFrame(features, columns=feature_cols)
    df['label'] = labels
    df['image_path'] = paths
    df = df[['image_path', 'label'] + feature_cols]
    df.to_csv(output_filename, index=False)
    print(f"Saved features to {output_filename}")

# --- Main Execution ---
if __name__ == '__main__':
    print("Setting up environment...")
    drive.mount('/content/drive')
    if not os.path.exists(OUTPUT_PATH):
        os.makedirs(OUTPUT_PATH)
    print(f"Using device: {DEVICE}")

    # --- Copy data to local disk for stability and speed ---
    print(f"\nCopying FER dataset from {DRIVE_BASE_PATH} to local Colab disk {LOCAL_DATA_PATH}...")
    if os.path.exists(LOCAL_DATA_PATH):
        shutil.rmtree(LOCAL_DATA_PATH)
    try:
        shutil.copytree(DRIVE_BASE_PATH, LOCAL_DATA_PATH)
        print("Dataset copied successfully. Using local files for all operations.")
    except Exception as e:
        print(f"ERROR: Could not copy dataset. Please check your DRIVE_BASE_PATH.")
        print(f"Ensure 'train' and 'test' folders are inside.")
        print(f"Details: {e}")
        exit()

    # Standard PyTorch transforms (used by ResNet, VGG, MobileNet)
    default_transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])

    # List of all 8 models to run
    MODELS_TO_RUN = [
        'resnet50',
        'retinanet',
        'vgg16',
        'mobilenetv3',
        'convnext',
        'efficientnetv2_m',
        'yolov8',
        'lbp'
    ]

    # --- Unified Loop for All 8 Models ---
    for model_name in MODELS_TO_RUN:
        print(f"\n" + "="*30)
        print(f"STARTING: {model_name.upper()} FEATURE EXTRACTION")
        print("="*30)

        X_train, y_train, paths_train = None, None, None
        X_test, y_test, paths_test = None, None, None
        file_prefix = f"{model_name}_features"

        if model_name == 'lbp':
            dataset_train = FERDataset(root_dir=os.path.join(LOCAL_DATA_PATH, 'train'), is_lbp=True)
            dataset_test = FERDataset(root_dir=os.path.join(LOCAL_DATA_PATH, 'test'), is_lbp=True)
            X_train, y_train, paths_train = extract_lbp_features(dataset_train)
            X_test, y_test, paths_test = extract_lbp_features(dataset_test)

        elif model_name == 'yolov8':
            dataset_train = FERDataset(root_dir=os.path.join(LOCAL_DATA_PATH, 'train'))
            dataset_test = FERDataset(root_dir=os.path.join(LOCAL_DATA_PATH, 'test'))
            X_train, y_train, paths_train = extract_yolo_features(dataset_train)
            X_test, y_test, paths_test = extract_yolo_features(dataset_test)

        else:
            # This block handles all 6 PyTorch models
            model, file_prefix, custom_transform = None, None, None
            if model_name == 'resnet50':
                model, file_prefix, custom_transform = create_resnet50_extractor()
            elif model_name == 'retinanet':
                model, file_prefix, custom_transform = create_retinanet_extractor()
            elif model_name == 'vgg16':
                model, file_prefix, custom_transform = create_vgg16_extractor()
            elif model_name == 'mobilenetv3':
                model, file_prefix, custom_transform = create_mobilenetv3_extractor()
            elif model_name == 'convnext':
                model, file_prefix, custom_transform = create_convnext_extractor()
            elif model_name == 'efficientnetv2_m':
                model, file_prefix, custom_transform = create_efficientnetv2_m_extractor()

            if model is None:
                print(f"Unknown model name: {model_name}. Skipping.")
                continue

            model.to(DEVICE)
            current_transform = custom_transform if custom_transform else default_transform
            print(f"Using transforms for input size: {current_transform.transforms[0].size}")

            # Process train set
            data_path_train = os.path.join(LOCAL_DATA_PATH, 'train')
            dataset_train = FERDataset(root_dir=data_path_train, transform=current_transform)
            if len(dataset_train) > 0:
                dataloader_train = DataLoader(dataset_train, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, collate_fn=collate_fn)
                if model_name == 'retinanet':
                    X_train, y_train, paths_train = extract_retinanet_features(dataloader_train, model)
                else:
                    X_train, y_train, paths_train = extract_dl_features(dataloader_train, model, model_name)

            # Process test set
            data_path_test = os.path.join(LOCAL_DATA_PATH, 'test')
            dataset_test = FERDataset(root_dir=data_path_test, transform=current_transform)
            if len(dataset_test) > 0:
                dataloader_test = DataLoader(dataset_test, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, collate_fn=collate_fn)
                if model_name == 'retinanet':
                    X_test, y_test, paths_test = extract_retinanet_features(dataloader_test, model)
                else:
                    X_test, y_test, paths_test = extract_dl_features(dataloader_test, model, model_name)

            del model
            torch.cuda.empty_cache()

        if X_train is None or X_test is None:
            print(f"Failed to extract features for {model_name}. Skipping.")
            continue

        # --- Standardize and Apply PCA ---
        print(f"\nApplying Standardization and PCA for {model_name}...")
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        X_test_scaled = scaler.transform(X_test)

        current_n_features = X_train_scaled.shape[1]

        # --- LOGIC TO STANDARDIZE TO 2048 ---
        if current_n_features == TARGET_FEATURES:
            print(f"{model_name} already has {TARGET_FEATURES} features. Saving as is.")
            X_train_final = X_train_scaled
            X_test_final = X_test_scaled
            final_n_components = TARGET_FEATURES

        elif current_n_features < TARGET_FEATURES:
            print(f"Skipping PCA for {model_name}: It only has {current_n_features} features (which is less than {TARGET_FEATURES}).")
            X_train_final = X_train_scaled
            X_test_final = X_test_scaled
            final_n_components = current_n_features
        else:
            # This applies to models with > 2048 features, like VGG16 (4096)
            print(f"Applying PCA... Reducing {current_n_features} features to {TARGET_FEATURES}.")
            pca = PCA(n_components=TARGET_FEATURES, random_state=RANDOM_STATE)
            X_train_final = pca.fit_transform(X_train_scaled)
            X_test_final = pca.transform(X_test_scaled)

            explained_variance = np.sum(pca.explained_variance_ratio_)
            print(f"PCA complete. Explained variance: {explained_variance:.4f}")
            final_n_components = TARGET_FEATURES

        # --- Save the new, standardized CSV files ---
        save_csv(X_train_final, y_train, paths_train, file_prefix, 'train', final_n_components)
        save_csv(X_test_final, y_test, paths_test, file_prefix, 'test', final_n_components)

        print(f"----- FINISHED: {model_name.upper()} -----")
        del X_train, y_train, X_test, y_test, X_train_scaled, X_test_scaled, X_train_final, X_test_final

    # --- Final Cleanup ---
    print("\nCleaning up local data...")
    try:
        shutil.rmtree(LOCAL_DATA_PATH)
        print("Local data copy removed.")
    except Exception as e:
        print(f"Warning: Could not remove local data folder. Error: {e}")

    print("\n\nAll feature extraction processes complete.")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Setting up environment...
Mounted at /content/drive
Using device: cpu

Copying FER dataset from /content/drive/MyDrive/FER to local Colab disk /content/FER_local...
Dataset copied successfully. Using local files for all operations.

STARTING: RESNET50 FEATURE EXTRACTION
Loading ResNet-50...
Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 103MB/s] 


Using transforms for input size: (224, 224)
Found 12436 valid image files in /content/FER_local/train.


Extracting resnet50 Features: 100%|██████████| 389/389 [56:34<00:00,  8.73s/it]


Found 2511 valid image files in /content/FER_local/test.


...


Applying Standardization and PCA for mobilenetv3...
Skipping PCA for mobilenetv3: It only has 1280 features (which is less than 2048).
Final features shape: (12436, 1280)
Saved features to /content/drive/MyDrive/FER/output_standardized_pca2048/mobilenetv3_features_train.csv
Final features shape: (2511, 1280)
Saved features to /content/drive/MyDrive/FER/output_standardized_pca2048/mobilenetv3_features_test.csv
----- FINISHED: MOBILENETV3 -----

STARTING: CONVNEXT FEATURE EXTRACTION
Loading ConvNeXt-Base...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


model.safetensors:   0%|          | 0.00/354M [00:00<?, ?B/s]

Using transforms for input size: 256
Found 12436 valid image files in /content/FER_local/train.


Extracting convnext Features:   1%|          | 2/389 [00:43<2:22:02, 22.02s/it]